# Classification Trees and Ensembles — Concepts

Read the idea, the formula and the small example in that order. The demonstration code is provided.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import confusion_matrix, roc_auc_score

## 1. Read a classification tree

An internal node asks a yes/no question. The **root** is the first node, and a **leaf** is a final node. Depth is the longest root-to-leaf path, counted in edges.

A leaf estimates class probabilities from the training class shares:
$$\hat P(y=k\mid\mathrm{leaf})=\frac{\mathrm{number\ in\ class}\ k}{\mathrm{number\ in\ leaf}}.$$
The predicted class is the class with the highest share. With 145 survivors and 125 non-survivors, $P(\mathrm{survived})=145/270\approx0.537$, so the predicted class is “survived”. A tree predicts probabilities as well as labels.

## 2. Gini: how mixed is a node?

For class shares $p_k$:
$$G=1-\sum_k p_k^2.$$
For two classes, $G=1-p^2-(1-p)^2$. A pure node has Gini 0; an equal 50/50 mixture has Gini 0.5. Entropy is an alternative:
$$H=-\sum_k p_k\log(p_k),$$
where a zero share contributes zero. Both favour purer children.

A split is judged by its size-weighted child impurity:
$$\Delta G=G_{\mathrm{parent}}-\left(\frac{n_L}{n}G_L+\frac{n_R}{n}G_R\right).$$
$n_L,n_R$ are child sizes; $n=n_L+n_R$. Pick the largest decrease, then repeat recursively. Worked example: the parent has 40 survivors and 60 non-survivors ($G=0.48$). The “male?” split gives weighted Gini 0.36 and decrease 0.12; the alternative first-class split gives a smaller decrease.

In [ ]:
p = 0.4
print('Parent Gini:', 1 - p ** 2 - (1 - p) ** 2)
g_left = 1 - 0.2 ** 2 - 0.8 ** 2
g_right = 1 - 0.7 ** 2 - 0.3 ** 2
weighted = 0.6 * g_left + 0.4 * g_right
print('Weighted child Gini:', weighted)
print('Decrease:', 0.48 - weighted)

## 3. Grow, stop and prune

Try candidate inputs and thresholds, keep the largest impurity decrease, and repeat in the children. Stop when a node is pure or a setting such as `max_depth` or `min_samples_leaf` prevents splitting.

Pruning adds a penalty per leaf, just as for regression trees:
$$\mathrm{objective}=\mathrm{tree\ impurity}+\alpha\times\mathrm{number\ of\ leaves}.$$
Larger `ccp_alpha` can remove branches. Deep trees may overfit; choose tree settings with CV, not training accuracy alone.

## 4. Random forest and boosting

A random forest builds many trees from bootstrap samples and random feature subsets at each split. Scikit-learn averages class probabilities:
$$\hat p=\frac1B\sum_{b=1}^B\hat p_b.$$
For 0.8, 0.6, 0.3 the mean is about 0.567: predict survived at threshold 0.5. Classic bagging is often described as voting; averaging class probabilities can give a different result from voting on tree labels.

Gradient boosting builds small trees sequentially to improve the current predictions. For classification it works with log-loss and pseudo-residuals, not simply the 0/1 errors. The principles are the same as in regression, but the outputs describe class probabilities.

In [ ]:
tree_probabilities = np.array([0.8, 0.6, 0.3])
print('Forest probability:', tree_probabilities.mean())
print('Predicted class:', int(tree_probabilities.mean() >= 0.5))

## 5. Putting it together: a small GridSearchCV

The final example combines train/test splitting, CV, a forest and AUC. `GridSearchCV` tries a small list of settings and keeps the one with the highest average validation score.

In the exercise the entire search is provided: two `max_features` values and three `min_samples_leaf` values, evaluated with 5-fold CV on the training rows. You explain the result rather than writing the search. Only after the search do we evaluate on test rows. AUC uses probabilities; a confusion matrix uses labels at a threshold.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))